# 02. VRFB GraphRAG: Performance & Failure Analysis Q&A

This notebook demonstrates a proof-of-concept (PoC) GraphRAG system for VRFB.

**Workflow:**
1. Retrieve context from Neo4j
2. Inject the domain-specific graph context into the LLM prompt
3. Generate answers using LLM based on causal relationships from the context

## Setup and Connections

In [ ]:
# Environment Setup and Imports
%pip install -q -r requirements.txt

from neo4j import GraphDatabase
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load Credentials
load_dotenv()
URI = os.getenv('NEO4J_URI')
USER = os.getenv('NEO4J_USER')
PASSWORD = os.getenv('NEO4J_PASSWORD')
BASE_URL = os.getenv('LLM_URL')
API_KEY = os.getenv('LLM_API_KEY')

# Connect to database
AUTH = (USER, PASSWORD)
driver = GraphDatabase.driver(URI, auth=AUTH)

# LLM connection
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)
LLM_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

## Full Graph Retrieval

In [2]:
def retrieve_all_knowledge():
    ''' Retrieve all nodes and relationships from the database '''

    query = """
    MATCH (n)
    OPTIONAL MATCH (n)-[r]->(m)
    RETURN n.name AS start_node, type(r) AS relation_type, m.name AS end_node
    """
    
    context = []
        
    with driver.session() as session:
        result = session.run(query)
        
        for record in result:
            # components with connections
            if record["relation_type"] and record["end_node"]:
                line = f"[{record['start_node']}] --({record['relation_type']})--> [{record['end_node']}]"
                context.append(line)
            # stand alone points without connections
            elif record["start_node"]:
                line = f"[{record['start_node']}]"
                context.append(line)
                
    # remove duplicates and combine all the text
    unique_lines = list(set(context))
    context_str = "The following is the complete knowledge graph data of vanadium redox flow battery:\n" + "\n".join(unique_lines)
    
    return context_str

print(retrieve_all_knowledge())

The following is the complete knowledge graph data of vanadium redox flow battery:
[Short Circuit] --(CAUSES)--> [Safety Issue]
[Round-Trip Energy Efficiency Trade-off]
[Unit Cell] --(CONTAINS)--> [Frame]
[Low Membrane Thickness] --(CAUSES)--> [High Crossover]
[Carbon Corrosion] --(CAUSES)--> [High Internal Resistance]
[Carbon Corrosion] --(CAUSES)--> [Flow Blockage]
[Catholyte Aged Charge State] --(CONTAINS)--> [V5+]
[V3+] --(REDUCED_TO)--> [V2+]
[Flow Sensor]
[High Frame Thickness] --(CAUSES)--> [Low Compression]
[High Membrane Thickness] --(CAUSES)--> [High Internal Resistance]
[V5+ Precipitation] --(CAUSES)--> [Flow Blockage]
[Membrane Burst] --(CAUSES)--> [Electrolyte Mixing]
[V3+] --(OXIDIZED_TO)--> [V4+]
[High Flowrate] --(CAUSES)--> [Low Internal Resistance]
[V4+] --(REDUCED_TO)--> [V3+]
[Flow Battery System] --(CONTAINS)--> [Temperature Sensor]
[Membrane & Separator] --(HAS_PROPERTIES)--> [Low Membrane Thickness]
[High Voltage Efficiency] --(HAS_IMPACT_ON)--> [Round-Trip Energ

In [3]:
def ask_battery_expert(user_question):
    '''  Ask the battery expert LLM a question based on the knowledge graph context. '''
    
    graph_context = retrieve_all_knowledge()
    
    # LLM should only use the graph context to answer the question
    prompt = f"""You are a vanadium redox flow battery expert, and you have the following knowledge graph context:{graph_context}. Please answer the following question BASED ON THE KNOWLEDGE GRAPH CONTEXT. You don't have to show the user the original nodes and relationships, just give your conclusions. If the question is not related to the knowledge graph context, please answer "I don't know". User question: {user_question}
    """
    
    # Call LLM
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

# test
question = 'What might be the root cause of high internal resistance and what does it lead to?'
print(ask_battery_expert(question))

**Root causes of high internal resistance (based on the knowledge graph):**  
- **High membrane thickness** – directly causes high internal resistance.  
- **Carbon corrosion** – produces high internal resistance.  
- **Electrode masking** – leads to high internal resistance.  
- **Low flow rate** – results in high internal resistance.  

**Consequences of high internal resistance:**  
- **Low voltage efficiency** (high internal resistance → low voltage efficiency).  
- Low voltage efficiency negatively affects the **round‑trip energy‑efficiency trade‑off** (via its impact on that trade‑off).  

Thus, factors such as a thick membrane, corroded carbon components, electrode masking, or insufficient flow can raise internal resistance, which in turn reduces voltage efficiency and overall energy‑efficiency performance of the vanadium redox flow battery.


-----

## Subgraph Retrieval

In [4]:
# Get all node names from the graph
def get_all_node_names(session):
    result = session.run("MATCH (n) WHERE n.name IS NOT NULL RETURN n.name AS name")
    return [r["name"] for r in result]

def link_entities(question, node_names):
    """
    Naive entity linking: find known node names that appear in the question.
    Falls back to token overlap if no exact substring is found.
    """
    q = question.lower()
    name_match = [name for name in node_names if name.lower() in q]
    if name_match:
        return name_match

    # fallback: rank nodes by how many of their words appear in the question
    q_tokens = set(q.replace("?", " ").split())
    scored = []
    for name in node_names:
        overlap = len(set(name.lower().split()) & q_tokens)
        if overlap:
            scored.append((overlap, name))
    scored.sort(reverse=True)
    return [name for _, name in scored[:3]]

In [5]:
def retrieve_subgraph(session, seed_names, max_hops=2):
    """
    From the entry nodes, walk up to max_hops and return only that subgraph as (start)-[rel]->(end) triples.
    """
    hops = int(max_hops)
    query = f"""
    MATCH (seed) WHERE seed.name IN $seeds
    MATCH path = (seed)-[*1..{hops}]-(other)
    UNWIND relationships(path) AS rel
    RETURN DISTINCT startNode(rel).name AS start, type(rel) AS rel_type, endNode(rel).name AS end
    """
    results = session.run(query, seeds=seed_names)
    return [f"[{result['start']}] --({result['rel_type']})--> [{result['end']}]" for result in results]


def retrieve_causal_chain(session, seed_names, max_hops=3):
    """
    Failure-analysis: separate ROOT CAUSES (what points into the seed) from CONSEQUENCES (what the seed points to). 
    Directional traversal maps directly onto 'why did X happen / what does X lead to'.
    """
    hops = int(max_hops)
    causes = session.run(f"""
        MATCH (cause)-[r*1..{hops}]->(seed)
        WHERE seed.name IN $seeds
        UNWIND r AS rel
        RETURN DISTINCT startNode(rel).name AS start, type(rel) AS rel_type, endNode(rel).name AS end
    """, seeds=seed_names)
    effects = session.run(f"""
        MATCH (seed)-[r*1..{hops}]->(effect)
        WHERE seed.name IN $seeds
        UNWIND r AS rel
        RETURN DISTINCT startNode(rel).name AS start, type(rel) AS rel_type, endNode(rel).name AS end
    """, seeds=seed_names)
    fmt = lambda rows: [f"[{x['start']}] --({x['rel_type']})--> [{x['end']}]" for x in rows]
    return {"causes": fmt(causes), "effects": fmt(effects)}

In [6]:
def ask_graphrag(question, max_hops=2):
    with driver.session() as session:
        names = get_all_node_names(session)
        seeds = link_entities(question, names)
        if not seeds:
            return "No relevant entity found in the knowledge graph for this question."

        # check if the question is about causes or effects
        if any(word in question.lower() for word in ["cause", "root cause", "why", "reason", "effect", "consequence", "lead to", "result", "result in"]):
            chain = retrieve_causal_chain(session, seeds, max_hops=max_hops)
            causes = "\n".join(sorted(set(chain["causes"]))) or "No causes found."
            effects = "\n".join(sorted(set(chain["effects"]))) or "No effects found."
            context = f"Causes:\n{causes}\n\nEffects:\n{effects}"
        else:
            triples = retrieve_subgraph(session, seeds, max_hops=max_hops)
            context = "\n".join(sorted(set(triples)))
            
        prompt = f"""You are a Vanadium Redox Flow Battery expert. 
                    Answer the question using ONLY the retrieved causal subgraph below.
                    If the subgraph does not contain the answer, say so.

                    Matched entities: {seeds}
                    Retrieved subgraph (relevant to the question): {context}

                    Question: {question}
                """

        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

In [7]:
if __name__ == "__main__":
    question = 'What might be the root cause of high internal resistance and what does it lead to?'
    print("Question:", question, "\n")
    print("Answer:", ask_graphrag(question, max_hops=2))

Question: What might be the root cause of high internal resistance and what does it lead to? 

Answer: **Possible root causes of high internal resistance (as indicated in the subgraph):**  
- **Carbon Corrosion** (which itself is caused by **High Voltage**)  
- **Electrode Masking** (which is caused by **Bubble Formation**)  
- **High Membrane Thickness**  
- **Low Flowrate**  

**What high internal resistance leads to:**  
- **[High Internal Resistance]** --(LEADS_TO)--> **[Low Voltage Efficiency]**  
- **[Low Voltage Efficiency]** --(HAS_IMPACT_ON)--> **[Round‑Trip Energy Efficiency Trade‑off]**  

Thus, any of the root causes above can give rise to high internal resistance, which in turn reduces voltage efficiency and affects the round‑trip energy efficiency trade‑off.


In [8]:
if __name__ == "__main__":
    question = 'What are the components of a Vanadium Redox Flow Battery?'
    print("Question:", question, "\n")
    print("Answer:", ask_graphrag(question, max_hops=2))

Question: What are the components of a Vanadium Redox Flow Battery? 

Answer: Based solely on the retrieved causal subgraph, the components of a Vanadium Redox Flow Battery (Flow Battery System) are:

- Battery Management System  
- Battery Stack  
- Cooling System  
- Flow Sensor  
- Pipe  
- Pump  
- Tank_Anolyte  
- Tank_Catholyte  
- Temperature Sensor  
- Unit Cell  

These are derived directly from the `--(CONTAINS)-->` relationships in the subgraph, which define the part-whole composition of the Flow Battery System. No other entities in the subgraph represent physical components (e.g., causal states like "High Flow Resistance" or "Flow Blockage" are excluded as they describe conditions, not constituent parts).  

If the subgraph did not contain this information, I would have stated so—but it explicitly lists all above components via containment links.
